# Issuance patterns

In [1]:
# Imports
from cryptography.hazmat.backends import default_backend
from cryptography.x509.oid import ExtensionOID
from cryptography.x509.oid import NameOID
import plotly.graph_objects as go
from cryptography import x509
import pandas as pd
import collections
import itertools
import datetime
import json

In [2]:
# Load certificates
certs = list()
with open("../data/certificates/certificates.json", "r") as f:
    for line in f:
        cert_str = json.loads(line)
        pem_bytes = cert_str.encode('utf-8')
        cert = x509.load_pem_x509_certificate(pem_bytes, default_backend())
        certs.append(cert)

In [3]:
# Find all CAs per IP
ip_ca = collections.defaultdict(list)
ip_cert_dates = collections.defaultdict(list)

for cert in certs:
    # Find the CA
    for attribute in cert.issuer:
        if attribute.oid == NameOID.ORGANIZATION_NAME:
            issuer_org = attribute.value
    # Get SAN IPs
    san = cert.extensions.get_extension_for_oid(ExtensionOID.SUBJECT_ALTERNATIVE_NAME).value
    for identifier in san:
        if isinstance(identifier,x509.IPAddress):
            ip_ca[str(identifier.value)].append((cert.not_valid_before_utc,issuer_org))
            ip_cert_dates[str(identifier.value)].append((cert.not_valid_before_utc,cert.not_valid_after_utc))

## Issuance patterns

In [4]:
# How many certificates each IP has issued for in 2025
ip_ca_count = [(i,len(ip_ca[i])) for i in ip_ca]
ip_ca_count = sorted(ip_ca_count, key=lambda x: x[1], reverse=True)

# Certificate counts alone
ca_count = [len(ip_ca[i]) for i in ip_ca]

print(f"IP addresses with the highest number of certificates:")
for ip,count in ip_ca_count[:10]:
    cas = set([i[1] for i in ip_ca[ip]])
    print(f"  {ip}: {count:,} ({', '.join(cas)})")


IP addresses with the highest number of certificates:
  172.65.247.74: 88,597 (Let's Encrypt)
  172.65.189.205: 88,268 (Let's Encrypt)
  158.69.34.226: 2,770 (ZeroSSL)
  23.246.59.164: 907 (Google Trust Services)
  2a00:86c0:1001:1059::164: 907 (Google Trust Services)
  23.246.59.163: 637 (Google Trust Services)
  2a00:86c0:1001:1059::163: 637 (Google Trust Services)
  2405:200:1606:600:49:44:188:aa: 286 (Google Trust Services)
  49.44.188.170: 286 (Google Trust Services)
  2405:200:1605:600:49:44:220:4a: 283 (Google Trust Services)


## Multi-CA IP addresses

In [5]:
# Check how many IPs have multiple CAs
multi_ca_2, multi_ca_more = list(), list()
for ip in ip_ca:
    cas = set([i[1] for i in ip_ca[ip]])
    if len(cas) > 1:
        multi_ca_2.append([ip,ip_ca[ip]])
        if len(cas) > 2:
            multi_ca_more.append([ip,ip_ca[ip]])

print(f"{len(multi_ca_2)} out of {len(ip_ca):,} IPs have 2 or more CAs ({len(multi_ca_more)} IPs have 3 or more CAs)")
print(f"---")

# Sort the list of issued certificates for each IP
transition_case_1, transition_case_2, transition_case_3 = list(), list(), list()
transition_3_sorted_ts = dict()
for ip,certificates in multi_ca_2:
    certificates_sorted = sorted(certificates)
    # Sort by the date of certificate issuance
    cas_sorted = [i[1] for i in certificates_sorted]
    # Remove consequtive same entries
    cas_deduplicated = [key for key, _ in itertools.groupby(cas_sorted)]
    
    # Find all the possible cases of IP change
    if len(set(cas_deduplicated)) == 2 and len(cas_deduplicated) == 2:
        transition_case_1.append((ip,cas_deduplicated))
    elif len(set(cas_deduplicated)) == 2 and len(cas_deduplicated) > 2:
        transition_case_3.append((ip,cas_deduplicated))
        transition_3_sorted_ts[ip] = [(i[0].strftime("%Y-%m-%d"),i[1]) for i in certificates_sorted]
    else:
        transition_case_2.append((ip,cas_deduplicated))

print(f"Case 1: {len(transition_case_1)} IPs have the A->B transition")
print(f"  Mostly transited to: {collections.Counter([i[1][1] for i in transition_case_1]).most_common(n=3)}")
print(f"")
print(f"Case 2: {len(transition_case_2)} IPs have the A->B->C transition")
for ip,cas in transition_case_2:
    print(f"  {ip}: {cas}")
print(f"")
print(f"Case 3: {len(transition_case_3)} IPs have the A->B->A transition")
for ip,cas in transition_case_3:
    print(f"  {ip}: {cas}")
    for cert in transition_3_sorted_ts[ip]:
        print(f"    {cert}")

805 out of 227,854 IPs have 2 or more CAs (7 IPs have 3 or more CAs)
---
Case 1: 791 IPs have the A->B transition
  Mostly transited to: [('Sectigo Limited', 372), ('ZeroSSL', 146), ("Let's Encrypt", 142)]

Case 2: 7 IPs have the A->B->C transition
  38.147.180.194: ['ZeroSSL', 'SSL Corporation', 'Unizeto Technologies S.A.']
  38.147.181.194: ['ZeroSSL', 'SSL Corporation', 'Unizeto Technologies S.A.']
  202.214.97.26: ['ZeroSSL', 'SSL Corporation', "Let's Encrypt"]
  8.153.206.15: ['ZeroSSL', 'SSL Corp', "Let's Encrypt"]
  62.234.85.234: ['ZeroSSL', 'SSL Corporation', "Let's Encrypt"]
  47.102.84.212: ['ZeroSSL', 'ZoTrus Technology Limited', 'DigiCert, Inc.']
  180.76.164.37: ['CerSign Technology Limited', 'SSL Corporation', 'SSL Corp']

Case 3: 7 IPs have the A->B->A transition
  117.72.175.190: ["Let's Encrypt", 'ZeroSSL', "Let's Encrypt"]
    ('2025-12-23', "Let's Encrypt")
    ('2025-12-23', "Let's Encrypt")
    ('2025-12-25', "Let's Encrypt")
    ('2025-12-27', 'ZeroSSL')
    ('20

In [6]:
# Plot the transition diagram for Case 1
transition_case_1_values = [i[1] for i in transition_case_1]
data = collections.Counter([tuple(i[1]) for i in transition_case_1]).most_common()
data = [[i[0][0], i[0][1], i[1]] for i in data]

df = pd.DataFrame(data, columns=['Source', 'Target', 'Value'])

left_labels = df['Source'].unique().tolist()
right_labels = df['Target'].unique().tolist()

left_indices = {label: i for i, label in enumerate(left_labels)}
right_indices = {label: i for i, label in enumerate(right_labels)}

right_indices_offset = {k: v + len(left_labels) for k, v in right_indices.items()}

all_nodes = left_labels + right_labels

sources, targets, values = list(), list(), list()

for _, row in df.iterrows():
    s_name = row['Source']
    t_name = row['Target']
    s_idx = left_indices[s_name]
    t_idx = right_indices_offset[t_name]
    sources.append(s_idx)
    targets.append(t_idx)
    values.append(row['Value'])

for label in left_labels:
    if label in right_indices_offset:
        sources.append(left_indices[label])
        targets.append(right_indices_offset[label])
        values.append(0)

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color='black', width=0.5),
        label=all_nodes,
        color=[
            '#C76D5B' if i < len(left_labels) else '#4F8A8B'
            for i in range(len(all_nodes))
        ]
    ),
    link=dict(source=sources, target=targets, value=values, color='rgba(200, 200, 200, 0.4)')
)])

fig.update_layout(font_size=15, width=800, height=1500, margin = {'l':1,'r':1,'t':1,'b':10})

fig.write_image("../data/figures/ca_transitions.pdf")
fig.show()